In [ ]:
import numpy as np

# 1. Capture Kaggle's native, pre-compiled NumPy version programmatically
native_numpy = np.__version__
print(f"Locking pipeline to native Kaggle NumPy: {native_numpy}")

# 2. Inject the variable directly into the shell install using IPython interpolation ($)
!pip install -q cellxgene-census "fsspec==2025.3.0" "numpy==$native_numpy"

In [ ]:
#!/usr/bin/env python3
"""
Project 13 — Path A, Part 2: identity_gap validation via CZ CELLxGene census.
Run on Kaggle (CPU is enough; the bottleneck is network I/O to census, not
compute). Companion: docs/TASK_pathA_kaggle.md, docs/pathA_part2_shard_plan.md
(READ THIS FIRST -- runtime estimate + shard-plan rationale), Part 1
(kaggle_pathA_part1_decodability_map.py, must be run first).

WHAT THIS VALIDATES
--------------------
Part 1 ranked all 201 Decima cell types by confound-controlled decodability
using Decima's own shipped `test_pearson` -- a proxy that is entirely
INTERNAL to Decima (held-out genes, but still Decima's own predictions vs
Decima's own training data). Part 2 asks the harder question: does the same
ranking hold against INDEPENDENTLY MEASURED expression, pulled fresh from
the CZ CELLxGene census? This reuses the `identity_gap` definition from
scripts 06/10 (measured cell-type-specificity z-score minus Decima-predicted
specificity z-score), now computed atlas-wide instead of just immune.

SCOPE (see docs/pathA_part2_shard_plan.md for the full rationale)
-----------------------------------------------------------------
A VALIDATION SAMPLE, not full atlas parity: 182/201 Decima cell types
matched to a census `cell_type` label -- 93 exact-string matches + 89
recovered via normalized/substring/fuzzy matching + a handful of manually-
reasoned prefix-pattern overrides (data/processed/pathA_part2_celltype_targets.csv,
has a `match_type` column: exact/fuzzy). 19 cell types were left
deliberately unmatched (ambiguous names with >=2 plausible census parents,
anatomical-region names reused as cell-type labels, or no matching concept
in this census snapshot at all) -- see the shard plan doc for the full
list; NOT force-matched. Pulls n_cells=800 per cell type (or all available
if fewer), split into two random halves of 400 for a reproducibility
check, one query per cell type against Decima's 18,457-gene set only (not
the full ~60k-gene transcriptome -- keeps the pull volume down).

RUNTIME
-------
Sandbox pilot (see docs/pathA_part2_shard_plan.md): ~30s fixed overhead +
~1.44e-5 s per (gene x cell) unit per query => ~4.1 min/cell type =>
~12.4h for all 182 cell types + ~30min aggregation => ~12.9h total. OVER
the 10h Kaggle cap, so per docs/TASK_pathA_kaggle.md's rule this ships as
**2 shards** (N_SHARDS=2 below, ~91 cell types / ~6.2h each -- comfortable
margin under 10h given sandbox-vs-Kaggle network uncertainty). Run once
with SHARD_INDEX=0, once more with SHARD_INDEX=1 (fresh session or same
one). CHECKPOINTED: each cell type's result is written to
part2_results/<safe_name>.parquet as it completes; re-running skips
completed cell types, so an interrupted run or a session that runs out of
time mid-shard resumes for free / loses at most one in-flight cell type.

Usage
-----
    N_SHARDS = 2         # default: 2 shards, ~91 cell types / ~6.2h each
    SHARD_INDEX = 0       # which shard this session covers (0-indexed);
                           # run once with 0, once more with 1
    python notebooks/kaggle_pathA_part2_identity_gap_validation.py
    python notebooks/kaggle_pathA_part2_identity_gap_validation.py --aggregate-only
        # light final step (run once, after both shards are done): merge
        # all part2_results/*.parquet + Part 1's map, re-run the law test
        # with identity_gap as the validation axis.
"""

from __future__ import annotations
import argparse, json, re, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# ==================== PATH & ENVIRONMENT CONFIG ====================
IS_KAGGLE = Path("/kaggle").exists()

if IS_KAGGLE:
    print("Scanning /kaggle/input to locate your files automatically...")
    
    # Track down the targets CSV anywhere inside the attached datasets
    target_file_matches = list(Path("/kaggle/input").rglob("pathA_part2_celltype_targets.csv"))
    
    if not target_file_matches:
        import os
        print(" CRITICAL: File not found. Printing directory tree of /kaggle/input to inspect:")
        count = 0
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                log(Path(root) / f)
                count += 1
                if count >= 30: break
            if count >= 30: break
        raise FileNotFoundError("Could not find 'pathA_part2_celltype_targets.csv' anywhere inside /kaggle/input.")

    # Dynamically set the input folders based on actual file locations
    PROC_IN = target_file_matches[0].parent
    print(f"-> Successfully mapped processed input directory to: {PROC_IN}")

    # Track down the raw metadata file anywhere inside the attached datasets
    metadata_matches = list(Path("/kaggle/input").rglob("metadata.h5ad"))
    if metadata_matches:
        RAW_IN = metadata_matches[0].parent
        print(f"-> Successfully mapped raw input directory to: {RAW_IN}")
    else:
        RAW_IN = PROC_IN.parent / "raw"
        log(f"-> metadata.h5ad not found via scan; falling back to relative sibling: {RAW_IN}")

    # Writable Outputs (must go to /kaggle/working)
    OUTPUT_REPO = Path("/kaggle/working")
    PROC_OUT = OUTPUT_REPO / "data" / "processed"
    RESULTS_DIR = OUTPUT_REPO / "part2_results"
else:
    # Local environment fallback
    REPO = Path(__file__).resolve().parent.parent
    RAW_IN = REPO / "data" / "raw"
    PROC_IN = REPO / "data" / "processed"
    PROC_OUT = PROC_IN
    RESULTS_DIR = REPO / "part2_results"

# Ensure only the writable output directories are created
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROC_OUT.mkdir(parents=True, exist_ok=True)
# ====================================================================

# ---- shard config: run once with SHARD_INDEX=0, once more with SHARD_INDEX=1 ----
N_SHARDS = 2
SHARD_INDEX = 1

N_CELLS_PER_TYPE = 800  # split into two halves of 400 client-side for split-half check
PILOT_FIXED_OVERHEAD_ESTIMATE_S = 30  # from the sandbox pilot; the live pilot below overwrites this


def log(m):
    print(f"[pathA-part2] {m}", flush=True)


def safe_name(ct: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", ct)[:80]


def load_targets():
    # FIXED: Read from Read-Only Input Directory
    df = pd.read_csv(PROC_IN / "pathA_part2_celltype_targets.csv")
    df = df.sort_values("decima_cell_type").reset_index(drop=True)
    my_slice = df.iloc[SHARD_INDEX::N_SHARDS] if N_SHARDS > 1 else df
    log(f"{len(df)} target cell types total; this shard ({SHARD_INDEX}/{N_SHARDS}) covers {len(my_slice)}")
    return my_slice


def load_decima_gene_set():
    import h5py
    # FIXED: Read from Read-Only Input Directory
    f = h5py.File(RAW_IN / "metadata.h5ad", "r")
    gene_id = f["var"]["gene_id"][:].astype(str)
    f.close()
    return gene_id


def load_decima_celltype_profiles():
    import h5py
    # FIXED: Read from Read-Only Input Directory
    f = h5py.File(RAW_IN / "metadata.h5ad", "r")
    preds = f["layers"]["preds"]
    gene_id = f["var"]["gene_id"][:].astype(str)
    obs = f["obs"]
    ct_g = obs["cell_type"]
    cats = np.array([x.decode() if isinstance(x, bytes) else x for x in ct_g["categories"][:]])
    codes = ct_g["codes"][:]
    cell_type_per_row = cats[codes]
    cell_types = sorted(set(cell_type_per_row))
    ct_index = {ct: i for i, ct in enumerate(cell_types)}
    ct_code = np.array([ct_index[ct] for ct in cell_type_per_row])
    n_genes = len(gene_id)
    sums = np.zeros((len(cell_types), n_genes), dtype=np.float64)
    counts = np.zeros(len(cell_types), dtype=np.int64)
    chunk = 800
    for start in range(0, preds.shape[0], chunk):
        end = min(start + chunk, preds.shape[0])
        block = preds[start:end, :].astype(np.float64)
        for i, c in enumerate(ct_code[start:end]):
            sums[c] += block[i]
            counts[c] += 1
    f.close()
    profile = sums / counts[:, None]
    return pd.DataFrame(profile, index=cell_types, columns=gene_id)


def pull_one_cell_type(census, census_cell_type: str, gene_id_list):
    import cellxgene_census

    meta = census["census_data"]["homo_sapiens"].obs.read(
        value_filter=f"cell_type == '{census_cell_type}' and is_primary_data == True",
        column_names=["soma_joinid", "dataset_id"],
    ).concat().to_pandas()
    
    if len(meta) == 0:
        return None, {"n_cells_available": 0}
        
    best_dataset = meta["dataset_id"].value_counts().idxmax()
    pool = meta[meta.dataset_id == best_dataset]
    n = min(N_CELLS_PER_TYPE, len(pool))
    rng = np.random.RandomState(hash(census_cell_type) % (2**31))
    sample_ids = pool["soma_joinid"].sample(n=n, random_state=rng).tolist()

    adata = cellxgene_census.get_anndata(
        census, organism="Homo sapiens",
        obs_coords=sample_ids,
        X_name="raw",
    )
    return adata, {"n_cells_available": len(meta), "n_cells_pulled": adata.n_obs, "dataset_id": best_dataset}


def cpm_log1p(adata, gene_id_order):
    import scipy.sparse as sp
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    lib = X.sum(axis=1, keepdims=True)
    lib[lib == 0] = 1
    cpm = X / lib * 1e6
    logx = np.log1p(cpm)
    var_ids = adata.var["feature_id"].values if "feature_id" in adata.var else adata.var_names.values
    idx_map = {g: i for i, g in enumerate(var_ids)}
    aligned = np.full((logx.shape[0], len(gene_id_order)), np.nan)
    for j, g in enumerate(gene_id_order):
        if g in idx_map:
            aligned[:, j] = logx[:, idx_map[g]]
    return aligned


def run_pulls():
    import cellxgene_census
    targets = load_targets()
    gene_id_list = load_decima_gene_set().tolist()

    log("live pilot: timing the first real pull to re-calibrate the ETA...")
    census = cellxgene_census.open_soma(census_version="2025-11-08")
    t0 = time.time()
    first_row = targets.iloc[0]
    first_decima_ct, first_census_ct = first_row["decima_cell_type"], first_row["census_cell_type"]
    adata0, meta0 = pull_one_cell_type(census, first_census_ct, gene_id_list)
    pilot_s = time.time() - t0
    remaining = len(targets) - 1
    eta_h = (pilot_s * remaining) / 3600
    log(f"  pilot cell type '{first_decima_ct}' -> census '{first_census_ct}': {pilot_s:.1f}s ({meta0}). "
        f"ETA for remaining {remaining} cell types in this shard: {eta_h:.2f}h")
    if eta_h > 9:
        log("  *** WARNING: projected >9h for this shard. Consider stopping now and splitting "
            "further (e.g. N_SHARDS=4) rather than risk an incomplete run. Checkpointing means "
            "no wasted work either way. ***")

    if adata0 is not None:
        aligned0 = cpm_log1p(adata0, gene_id_list)
        save_result(first_decima_ct, aligned0, meta0)

    for i in range(1, len(targets)):
        row = targets.iloc[i]
        decima_ct, census_ct = row["decima_cell_type"], row["census_cell_type"]
        out_path = RESULTS_DIR / f"{safe_name(decima_ct)}.parquet"
        if out_path.exists():
            log(f"  [{i+1}/{len(targets)}] {decima_ct}: already done, skipping")
            continue
        t0 = time.time()
        try:
            adata, meta = pull_one_cell_type(census, census_ct, gene_id_list)
            if adata is not None:
                aligned = cpm_log1p(adata, gene_id_list)
                save_result(decima_ct, aligned, meta)
            else:
                log(f"  [{i+1}/{len(targets)}] {decima_ct}: no cells found in census, skipping")
        except Exception as e:  # noqa: BLE001
            log(f"  [{i+1}/{len(targets)}] {decima_ct}: FAILED: {e}")
            continue
        log(f"  [{i+1}/{len(targets)}] {decima_ct}: {time.time()-t0:.1f}s")
    census.close()
    log("shard complete.")


def save_result(decima_cell_type, aligned_expr, meta):
    n = aligned_expr.shape[0]
    rng = np.random.RandomState(hash(decima_cell_type) % (2**31))
    idx = rng.permutation(n)
    half_a = np.nanmean(aligned_expr[idx[:n//2]], axis=0)
    half_b = np.nanmean(aligned_expr[idx[n//2:]], axis=0)
    full_mean = np.nanmean(aligned_expr, axis=0)
    out = pd.DataFrame({"mean_expr_full": full_mean, "mean_expr_half_a": half_a, "mean_expr_half_b": half_b})
    out_path = RESULTS_DIR / f"{safe_name(decima_cell_type)}.parquet"
    out.to_parquet(out_path)
    (RESULTS_DIR / f"{safe_name(decima_cell_type)}.meta.json").write_text(
        json.dumps({"cell_type": decima_cell_type, **meta}))


def aggregate():
    log("aggregating all part2_results/*.parquet + Part 1's map -> identity_gap validation")
    result_files = sorted(RESULTS_DIR.glob("*.parquet"))
    log(f"found {len(result_files)} completed cell-type pulls")
    if not result_files:
        log("nothing to aggregate yet.")
        return

    decima_profiles = load_decima_celltype_profiles()

    rows = []
    for fp in result_files:
        meta_path = fp.with_suffix("").with_suffix(".meta.json")
        meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
        ct = meta.get("cell_type", fp.stem.replace("_", " "))
        df = pd.read_parquet(fp)
        rows.append({"cell_type": ct, "mean_expr_full": df["mean_expr_full"].values,
                     "mean_expr_half_a": df["mean_expr_half_a"].values,
                     "mean_expr_half_b": df["mean_expr_half_b"].values})

    measured_cts = [r["cell_type"] for r in rows]
    full_mat = np.stack([r["mean_expr_full"] for r in rows])

    def zscore_rows(mat):
        mu = np.nanmean(mat, axis=0, keepdims=True)
        sd = np.nanstd(mat, axis=0, keepdims=True)
        sd[sd == 0] = np.nan
        return (mat - mu) / sd

    spec_measured = zscore_rows(full_mat)

    pred_mat = decima_profiles.reindex(measured_cts).values
    spec_pred = zscore_rows(pred_mat)

    identity_gap = spec_measured - spec_pred
    gap_summary = pd.DataFrame({
        "cell_type": measured_cts,
        "mean_abs_identity_gap": np.nanmean(np.abs(identity_gap), axis=1),
        "mean_identity_gap": np.nanmean(identity_gap, axis=1),
    })

    half_a_mat = np.stack([r["mean_expr_half_a"] for r in rows])
    half_b_mat = np.stack([r["mean_expr_half_b"] for r in rows])
    spec_a = zscore_rows(half_a_mat)
    spec_b = zscore_rows(half_b_mat)
    per_gene_splithalf = []
    for i in range(len(measured_cts)):
        valid = ~(np.isnan(spec_a[i]) | np.isnan(spec_b[i]))
        if valid.sum() > 10:
            r, _ = stats.pearsonr(spec_a[i][valid], spec_b[i][valid])
            per_gene_splithalf.append(r)
    log(f"per-cell-type measured-specificity split-half Pearson: "
        f"mean={np.nanmean(per_gene_splithalf):.3f} (n={len(per_gene_splithalf)} cell types)")

    # FIXED: Read part1_map from the Read-Only Input Directory
    part1_map = pd.read_parquet(PROC_IN / "atlas_celltype_decodability_map.parquet")
    merged = gap_summary.merge(part1_map, on="cell_type", how="left")
    
    # FIXED: Write validation mapping back to the Writable Output Directory
    merged.to_parquet(PROC_OUT / "atlas_identity_gap_validation.parquet")

    r_val, p_val = stats.spearmanr(merged["mean_abs_identity_gap"], -merged["decodability_controlled_mean"],
                                    nan_policy="omit")
    report = {
        "n_cell_types_validated": len(merged),
        "measured_specificity_splithalf_pearson_mean": float(np.nanmean(per_gene_splithalf)),
        "validation_test": {
            "note": "identity_gap (measured vs Decima) should be HIGHER (worse) exactly where "
                    "test_pearson-based decodability_controlled is LOWER, if Part 1's map reflects "
                    "something real and not a test_pearson-specific artifact.",
            "spearman_rho_gap_vs_neg_decodability": float(r_val), "p": float(p_val),
        },
    }
    # FIXED: Write output JSON report to the Writable Output Directory
    (PROC_OUT / "atlas_part2_validation_report.json").write_text(json.dumps(report, indent=2, default=str))
    log(f"VALIDATION: rho={r_val:.3f}, p={p_val:.3g} -- wrote atlas_identity_gap_validation.parquet + report")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--aggregate-only", action="store_true")
    args, unknown = ap.parse_known_args()
    if args.aggregate_only:
        aggregate()
    else:
        run_pulls()


if __name__ == "__main__":
    main()